# TMDB Movie Recommendation 

Movie recommendation system using the TMDB 6000 database available on kaggle: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

This script is considerably inspired by https://github.com/campusx-official/movie-recommender-system-tmdb-dataset

In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

In [3]:
data = movies.merge(credits, on='title')
data.shape

(4809, 23)

In [4]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

## Feature Selection
- ID (id)
- Title of the movie (title) (English translation of original_title)
- Genre of the movie (genres)
- Keywords associated with the movie (keywords)
- Overview of the movie (overview)
- Cast starred in the movie (cast)
- Crew that worked for the movie (crew)
- Date of movie release (release_date)

In [5]:
df = data[['id', 'title', 'genres', 'keywords', 'overview', 'cast', 'crew', 'release_date']]

# Data Cleaning

- Exclude rows with null values because they consist of less than < 1% of the data

In [6]:
df.isnull().sum()

id              0
title           0
genres          0
keywords        0
overview        3
cast            0
crew            0
release_date    1
dtype: int64

In [7]:
df.dropna(inplace=True)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [8]:
df.duplicated().sum()

0

## Cleaning up the genres and keywords column

The genres and keywords column is a string that is inplace for a list. The string will be converted to a list in Python. Then we extract the names from that list.

In [9]:
def reformat(row):
    import ast
    new_row = []
    for item in ast.literal_eval(row):
        new_row.append(item['name'])

    return new_row

In [10]:
df['genres'] = df['genres'].apply(reformat)
df['keywords'] =  df['keywords'].apply(reformat)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\3702656427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genres'] = df['genres'].apply(reformat)
C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\3702656427.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['keywords'] =  df['keywords'].apply(reformat)


In [11]:
df.head()

,id,title,genres,keywords,overview,cast,crew,release_date
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",2007-05-19
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",A cryptic message from Bond’s past sends him o...,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",2015-10-26
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",Following the death of District Attorney Harve...,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",2012-07-16
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","John Carter is a war-weary, former military ca...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",2012-03-07


## Cleaning up the cast column

The cast column is a string that is inplace for a list. The string will be converted to a list in Python. Then we extract the top 3 names from that list.

In [12]:
def reformat_cast(row):
    import ast
    new_row = []
    counter = 0
    for item in ast.literal_eval(row):
        if counter != 3:
            new_row.append(item['name'])
            counter+=1
        else:
            break

    return new_row

In [13]:
df['cast'] = df['cast'].apply(reformat_cast)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\1576142058.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cast'] = df['cast'].apply(reformat_cast)


## Cleaning up the crew column

The crew column is a string that is inplace for a list. The string will be converted to a list in Python. Then we extract the Director's names from that list.

In [14]:
def get_director(row):
    import ast
    new_row = []
    for item in ast.literal_eval(row):
        if item['job'] == 'Director':
            new_row.append(item['name'])
            break
    
    return new_row

In [15]:
df['crew'] = df['crew'].apply(get_director)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\4103153286.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['crew'] = df['crew'].apply(get_director)


In [16]:
df.head()

,id,title,genres,keywords,overview,cast,crew,release_date
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],2007-05-19
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],2015-10-26
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],2012-07-16
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],2012-03-07


## Tokenising the overview column

In [17]:
df['overview'] = df['overview'].apply(lambda x:x.split())

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\2950014412.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['overview'] = df['overview'].apply(lambda x:x.split())


## Discretising the release_date column

We want to discretise the release_date column so that we can split the dates into decades.

In [18]:
def release_decade(row):
    year = pd.to_datetime(row).year
    decade = int(np.floor(year/10) * 10)

    return decade

In [19]:
df['release_date'] = df['release_date'].apply(release_decade)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\1524738953.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['release_date'] = df['release_date'].apply(release_decade)


## Text Preprocessing

In [20]:
df['genres'] = df['genres'].apply(lambda x: [i.replace(" ", "") for i in x ])
df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ", "") for i in x ])

df['cast'] = df['cast'].apply(lambda x: [i.replace(" ", "") for i in x ])
df['crew'] = df['crew'].apply(lambda x: [i.replace(" ", "") for i in x ])

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\3431474976.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genres'] = df['genres'].apply(lambda x: [i.replace(" ", "") for i in x ])
C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\3431474976.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ", "") for i in x ])
C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\3431474976.py:4: SettingWithCopyWarning: 
A value is trying to be set on a c

In [21]:
df['tags'] = df['genres'] + df['overview'] + df['keywords'] + df['cast'] + df['crew']

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\4127071975.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['genres'] + df['overview'] + df['keywords'] + df['cast'] + df['crew']


In [22]:
df['tags'] = df['tags'].apply(lambda x: " ".join(x))
df['tags'] = df['tags'].apply(lambda x: x.lower())

C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\1778811128.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x: " ".join(x))
C:\Users\eshaa\AppData\Local\Temp\ipykernel_13268\1778811128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x: x.lower())


In [23]:
text_data = df[['id', 'title', 'tags']]

In [24]:
text_data 

,id,title,tags
0,19995,Avatar,action adventure fantasy sciencefiction in the...
1,285,Pirates of the Caribbean: At World's End,"adventure fantasy action captain barbossa, lon..."
2,206647,Spectre,action adventure crime a cryptic message from ...
3,49026,The Dark Knight Rises,action crime drama thriller following the deat...
4,49529,John Carter,action adventure sciencefiction john carter is...
...,...,...,...
4804,9367,El Mariachi,action crime thriller el mariachi just wants t...
4805,72766,Newlyweds,comedy romance a newlywed couple's honeymoon i...
4806,231617,"Signed, Sealed, Delivered","comedy drama romance tvmovie ""signed, sealed, ..."
4807,126186,Shanghai Calling,when ambitious new york attorney sam is sent t...
